# Hyperparameter Tuning

> Based on Stanford CS230 — C2M3, Andrew Ng / DeepLearning.AI

Hyperparameters control the model and training process — they are not learned from data but chosen by you. Finding good hyperparameters is an empirical, iterative process.

## Learning Objectives

1. Rank hyperparameters by importance and know where to start
2. Implement random search and compare it to grid search
3. Use log-scale sampling for scale-sensitive hyperparameters (e.g. learning rate)
4. Understand the coarse-to-fine tuning strategy

## Hyperparameter Priority

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 580 200" width="580" height="200" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Priority tiers -->
  <rect x="20"  y="30" width="540" height="44" rx="6" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.6"/>
  <rect x="20"  y="84" width="540" height="44" rx="6" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.6"/>
  <rect x="20"  y="138" width="540" height="44" rx="6" fill="#5b9bd5" fill-opacity="0.10" stroke="#5b9bd5" stroke-width="1.5"/>
  <!-- Tier labels -->
  <text x="30"  y="50" fill="#b03a3a" font-size="11" font-weight="bold">Priority 1:</text>
  <text x="110" y="50" fill="#555" font-size="11">Learning rate α</text>
  <text x="30"  y="67" fill="#888" font-size="10">Most important — tune this first. Spans orders of magnitude.</text>
  <text x="30"  y="104" fill="#a07010" font-size="11" font-weight="bold">Priority 2:</text>
  <text x="110" y="104" fill="#555" font-size="11">β (momentum), # hidden units, mini-batch size</text>
  <text x="30"  y="121" fill="#888" font-size="10">Second pass — after α is reasonable.</text>
  <text x="30"  y="158" fill="#3a7bbf" font-size="11" font-weight="bold">Priority 3:</text>
  <text x="110" y="158" fill="#555" font-size="11">Network depth L, learning rate decay, β₁ β₂ ε (Adam)</text>
  <text x="30"  y="175" fill="#888" font-size="10">Rarely need to change from defaults.</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Grid search vs Random search ----
# Hypothetical: performance depends mainly on lr (horizontal axis)
# Grid search: fixed points; random search: more unique lr values sampled

np.random.seed(7)
n = 25

# Grid search: 5x5 grid
grid_lr    = np.tile(np.linspace(0.001, 0.1, 5), 5)
grid_units = np.repeat(np.array([16, 32, 64, 128, 256]), 5)

# Random search: 25 random points
rand_lr    = 10 ** np.random.uniform(-3, -1, n)
rand_units = np.random.choice([16, 32, 48, 64, 96, 128, 192, 256], n)

# Simulated performance: mainly driven by lr (peak around 0.01)
def perf(lr, units):
    lr_score  = np.exp(-50*(np.log10(lr) - np.log10(0.01))**2)
    unit_score = 0.1 * np.log2(units) / np.log2(256)
    return lr_score + unit_score + np.random.randn() * 0.03

grid_perf = np.array([perf(lr, u) for lr, u in zip(grid_lr, grid_units)])
rand_perf = np.array([perf(lr, u) for lr, u in zip(rand_lr, rand_units)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Grid Search vs Random Search for Hyperparameter Tuning', fontsize=13, fontweight='bold')

for ax, lrs, units, perfs, title, color in [
    (axes[0], grid_lr, grid_units, grid_perf, 'Grid Search  (5×5 = 25 trials)', P[1]),
    (axes[1], rand_lr, rand_units, rand_perf, 'Random Search  (25 trials)', P[0]),
]:
    sc = ax.scatter(lrs, units, c=perfs, cmap='YlGn', s=120, zorder=3,
                    vmin=0, vmax=1.1, edgecolors='#555', lw=0.7)
    best_idx = np.argmax(perfs)
    ax.scatter(lrs[best_idx], units[best_idx], s=250, marker='*',
               color=P[2], zorder=5, label=f'Best  lr={lrs[best_idx]:.4f}  u={units[best_idx]}')
    plt.colorbar(sc, ax=ax, label='Performance')
    ax.set_xscale('log')
    ax.set_xlabel('Learning rate (log scale)')
    ax.set_ylabel('Hidden units')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

print("Unique learning rates sampled:")
print(f"  Grid search:   {len(set(grid_lr))} unique values")
print(f"  Random search: {len(rand_lr)} unique values")
print(f"\nBest performance:")
print(f"  Grid search:   {grid_perf.max():.3f}  at lr={grid_lr[grid_perf.argmax()]:.4f}")
print(f"  Random search: {rand_perf.max():.3f}  at lr={rand_lr[rand_perf.argmax()]:.4f}")


In [ ]:
# ---- Log-scale sampling ----
# Why sample α on a log scale, not linearly?
# Linear: np.random.uniform(0.0001, 1.0)  — 90% of samples in [0.1, 1.0]
# Log:    10 ** np.random.uniform(-4, 0)  — equal coverage per decade

np.random.seed(0)
n_samples = 5000

linear_samples = np.random.uniform(0.0001, 1.0, n_samples)
log_samples    = 10 ** np.random.uniform(-4, 0, n_samples)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Sampling Strategy for Learning Rate', fontsize=12, fontweight='bold')

bins = np.logspace(-4, 0, 40)
axes[0].hist(linear_samples, bins=bins, color=P[1], alpha=0.8)
axes[0].set_xscale('log')
axes[0].set_title('Linear Sampling — biased toward large values')
axes[0].set_xlabel('Learning rate (log scale)')
axes[0].set_ylabel('Count')
axes[0].grid(True)

axes[1].hist(log_samples, bins=bins, color=P[3], alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_title('Log Sampling — uniform coverage per decade')
axes[1].set_xlabel('Learning rate (log scale)')
axes[1].set_ylabel('Count')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# ---- Coarse-to-fine strategy ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.suptitle('Coarse-to-Fine Tuning Strategy', fontsize=12, fontweight='bold')

np.random.seed(2)
# Coarse: large region
c_lr = 10 ** np.random.uniform(-4, 0, 25)
c_units = np.random.choice([8, 16, 32, 64, 128, 256, 512], 25)
c_perf = np.array([perf(lr, u) for lr, u in zip(c_lr, c_units)])
best_lr_c, best_u_c = c_lr[c_perf.argmax()], c_units[c_perf.argmax()]

# Fine: zoom around best
f_lr = 10 ** np.random.uniform(np.log10(best_lr_c)-0.5, np.log10(best_lr_c)+0.5, 25)
f_units = np.random.randint(max(8, best_u_c//2), best_u_c*2+1, 25)
f_perf = np.array([perf(lr, u) for lr, u in zip(f_lr, f_units)])

axes[0].scatter(c_lr, c_units, c=c_perf, cmap='YlGn', s=100, vmin=0, vmax=1.1, zorder=3, edgecolors='#555', lw=0.7)
rect = patches.FancyBboxPatch((best_lr_c*0.3, best_u_c*0.5), best_lr_c*3, best_u_c,
                               boxstyle="round,pad=0.0", linewidth=2, edgecolor=P[2], facecolor='none', linestyle='--', zorder=5)
axes[0].add_patch(rect)
axes[0].set_xscale('log')
axes[0].set_xlabel('Learning rate'); axes[0].set_ylabel('Hidden units')
axes[0].set_title('Coarse search  →  identify promising region')
axes[0].grid(True)

axes[1].scatter(f_lr, f_units, c=f_perf, cmap='YlGn', s=120, vmin=0, vmax=1.1, zorder=3, edgecolors='#555', lw=0.7)
axes[1].set_xscale('log')
axes[1].set_xlabel('Learning rate'); axes[1].set_ylabel('Hidden units')
axes[1].set_title('Fine search  →  zoom into best region')
axes[1].grid(True)

plt.tight_layout()
plt.show()


## Panda vs Caviar Approach

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 150" width="600" height="150" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <rect x="20"  y="25" width="260" height="110" rx="6" fill="#f4b942" fill-opacity="0.12" stroke="#f4b942" stroke-width="1.5"/>
  <rect x="320" y="25" width="260" height="110" rx="6" fill="#5b9bd5" fill-opacity="0.12" stroke="#5b9bd5" stroke-width="1.5"/>
  <!-- Panda -->
  <text x="150" y="48" text-anchor="middle" fill="#a07010" font-size="12" font-weight="bold">Panda approach</text>
  <text x="150" y="65" text-anchor="middle" fill="#a07010" font-size="10">(babysit one model)</text>
  <text x="35"  y="86" fill="#555" font-size="10">✓ Limited compute</text>
  <text x="35"  y="102" fill="#555" font-size="10">✓ Monitor and nudge hyperparams daily</text>
  <text x="35"  y="118" fill="#555" font-size="10">✓ Typical for large models / small teams</text>
  <!-- Caviar -->
  <text x="450" y="48" text-anchor="middle" fill="#3a7bbf" font-size="12" font-weight="bold">Caviar approach</text>
  <text x="450" y="65" text-anchor="middle" fill="#3a7bbf" font-size="10">(train many models in parallel)</text>
  <text x="335" y="86" fill="#555" font-size="10">✓ Abundant compute (cloud)</text>
  <text x="335" y="102" fill="#555" font-size="10">✓ Many configs run simultaneously</text>
  <text x="335" y="118" fill="#555" font-size="10">✓ Pick the winner</text>
</svg>

> **Practical workflow:** Use random search on a log scale. Find a coarse region. Zoom in and repeat. Retrain when data or architecture changes — hyperparams may not transfer.
